# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Anoushka Yadav
**`Roll Number`:** U20230076
**`GitHub Branch`:** anoushka_U20230076

# Imports and Setup

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from rlcmab_sampler import sampler

# Seed for reproducibility
np.random.seed(42)

# Load Datasets

In [44]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [57]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# -----------------------------------
# 1. Load data (adjust filename)
# -----------------------------------

train_users = train_users.dropna().reset_index(drop=True)

# -----------------------------------
# 2. Drop missing values
# -----------------------------------

train_users = train_users.dropna().reset_index(drop=True)

# -----------------------------------
# 3. Encode target labels
# -----------------------------------

le = LabelEncoder()
train_users["user_class"] = le.fit_transform(train_users["label"])

# -----------------------------------
# 4. Separate features and target
# -----------------------------------

X = train_users.drop(columns=["label", "user_class"])
y = train_users["user_class"]

# -----------------------------------
# 5. Identify column types
# -----------------------------------

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

# -----------------------------------
# 6. Preprocessing pipeline
# -----------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), cat_cols)
    ]
)

# -----------------------------------
# 7. Train / Validation split
# -----------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------------
# 8. Class distribution check
# -----------------------------------

print("\nClass distribution:")
print(train_users["user_class"].value_counts(normalize=True))


Numeric columns: ['age', 'income', 'clicks', 'purchase_amount', 'session_duration', 'content_variety', 'engagement_score', 'num_transactions', 'avg_monthly_spend', 'avg_cart_value', 'browsing_depth', 'revisit_rate', 'scroll_activity', 'time_on_site', 'interaction_count', 'preferred_price_range', 'discount_usage_rate', 'wishlist_size', 'product_views', 'repeat_purchase_gap (days)', 'churn_risk_score', 'loyalty_index', 'screen_brightness', 'battery_percentage', 'cart_abandonment_count', 'background_app_count', 'session_inactivity_duration', 'network_jitter']
Categorical columns: ['user_id', 'browser_version', 'region_code']

Class distribution:
user_class
1    0.481567
0    0.434716
2    0.083717
Name: proportion, dtype: float64


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [58]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# -----------------------------------
# 1. Build XGBoost pipeline
# -----------------------------------

xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

# -----------------------------------
# 2. Fit with early stopping
# -----------------------------------

X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)

xgb_pipeline.named_steps["clf"].fit(
    X_train_transformed,
    y_train
)


# -----------------------------------
# 3. Predict
# -----------------------------------

y_pred = xgb_pipeline.named_steps["clf"].predict(X_val_transformed)

# -----------------------------------
# 4. Evaluation
# -----------------------------------

print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_val, y_pred, target_names=le.classes_))

# -----------------------------------
# 5. Feature importance (optional)
# -----------------------------------

importances = xgb_pipeline.named_steps["clf"].feature_importances_
print("\nTop 20 feature importances:")
print(np.sort(importances)[-20:])


Validation Accuracy: 0.9540229885057471

Classification Report:

              precision    recall  f1-score   support

      user_1       0.95      0.96      0.95       113
      user_2       0.98      0.96      0.97       126
      user_3       0.87      0.91      0.89        22

    accuracy                           0.95       261
   macro avg       0.93      0.94      0.94       261
weighted avg       0.95      0.95      0.95       261


Top 20 feature importances:
[0.00880854 0.00906577 0.00909668 0.00912475 0.00934172 0.00968764
 0.00970243 0.01122634 0.01143047 0.01192565 0.01257104 0.01511836
 0.04542482 0.05569249 0.07294962 0.08605477 0.08908296 0.08998151
 0.10505164 0.23229836]


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
